# Notebook 01 — Inspección inicial

## Planteamiento de trabajo

En esta primera etapa el objetivo es conocer el dataset y reunir evidencia sobre su estado, sin modificar los datos ni tomar decisiones de limpieza todavía. Esa evidencia es la que va a justificar las decisiones de las etapas siguientes.

El conjunto de datos reúne información de **usuarios de una plataforma de streaming**. Cada fila representa a un usuario e incluye datos demográficos, hábitos de consumo y uso del servicio.

**Variables del dataset:**

| Variable                 | Significado                      |
| ------------------------ | -------------------------------- |
| user_id                  | Identificador único del usuario  |
| age                      | Edad                             |
| subscription_plan        | Plan contratado                  |
| monthly_watch_time_mins  | Minutos vistos por mes           |
| country                  | País                             |
| favorite_genre           | Género favorito                  |
| last_login_date          | Último ingreso                   |
| customer_support_tickets | Cantidad de consultas al soporte |

**Qué vamos a revisar en esta inspección:** la estructura general y las dimensiones, los tipos de dato, los valores faltantes, los registros duplicados y las posibles inconsistencias, para detectar qué necesitará atención más adelante.



## Librerías

In [1]:
import pandas as pd
import numpy as np

Se importan las librerías base del análisis: **pandas** para manejar la tabla de datos y **numpy** para operaciones numéricas.

## Carga del dataset

In [2]:
df = pd.read_json("../data/raw/streaming_users_dirty.json")

print("Dataset cargado correctamente.")

Dataset cargado correctamente.


El dataset se cargó correctamente desde `data/raw/`. Trabajamos siempre sobre el archivo original, sin modificarlo.

## Vista previa: primeras filas

In [3]:
# Primeras filas del dataset

df.head()

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1


Las primeras filas confirman que cada registro es un usuario y que las columnas contienen lo esperado (edad, plan, país, género, etc.).

## Vista previa: últimas filas

In [4]:
# Últimas filas del dataset

df.tail()

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
8155,10923,23,Premium,1161.4,Colombia,Romance,2023-05-15,0
8156,16525,27,Básico,436.2,Uruguay,Documental,2018-09-06,4
8157,11222,13,Estándar,1321.8,México,Documental,2019-02-08,0
8158,15613,38,Estándar,835.7,Brasil,Drama,2022-02-05,0
8159,16912,25,Estándar,1468.7,Argentina,Romance,2022-08-12,3


Las últimas filas muestran la misma estructura que las primeras, lo que indica que el archivo es consistente de principio a fin.

## Dimensiones del dataset

In [5]:
# Cantidad de filas y columnas

filas, columnas = df.shape

print(f"Cantidad de filas: {filas}")
print(f"Cantidad de columnas: {columnas}")

Cantidad de filas: 8160
Cantidad de columnas: 8


El dataset tiene **8160 filas y 8 columnas**: 8160 usuarios descritos por 8 variables.

## Estructura general y tipos de dato

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8160 entries, 0 to 8159
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8160 non-null   int64  
 1   age                       8160 non-null   int64  
 2   subscription_plan         8160 non-null   str    
 3   monthly_watch_time_mins   7967 non-null   float64
 4   country                   8160 non-null   str    
 5   favorite_genre            7920 non-null   str    
 6   last_login_date           7840 non-null   str    
 7   customer_support_tickets  8160 non-null   int64  
dtypes: float64(1), int64(3), str(4)
memory usage: 510.1 KB


8 columnas y sus tipos (numéricas y de texto). Ya se notan **valores faltantes** en tres variables: tiempo de visualización, género favorito y fecha del último ingreso.

## Nombres de las columnas

In [7]:
df.columns

Index(['user_id', 'age', 'subscription_plan', 'monthly_watch_time_mins',
       'country', 'favorite_genre', 'last_login_date',
       'customer_support_tickets'],
      dtype='str')

Quedan listados los nombres de las 8 variables, útil para referirnos a ellas en el resto del trabajo.

## Tipo de dato por variable

In [8]:
df.dtypes

user_id                       int64
age                           int64
subscription_plan               str
monthly_watch_time_mins     float64
country                         str
favorite_genre                  str
last_login_date                 str
customer_support_tickets      int64
dtype: object

Confirmamos el tipo de cada variable. La fecha de último ingreso figura como texto (*object*), algo a corregir más adelante.

## Resumen estadístico (variables numéricas)

In [9]:
df.describe().round(2)

,user_id,age,monthly_watch_time_mins,customer_support_tickets
count,8160.00,8160.00,7967.00,8160.00
mean,13995.43,34.10,1107.35,1.80
std,2310.81,14.51,5310.44,11.33
min,10000.00,-5.00,-120.00,-1.00
25%,11987.75,25.00,489.20,0.00
50%,13998.50,33.00,757.40,1.00
75%,15997.25,42.00,1045.70,1.00
max,17999.00,150.00,99999.00,150.00


El resumen numérico revela **valores imposibles**: edades negativas y de hasta 150 años, tiempos de visualización negativos o enormes (99999) y cantidades negativas de tickets. Son señales de errores de carga que habrá que revisar.

## Resumen de las variables categóricas

In [10]:
df.describe(include="object")

C:\Users\mateo\AppData\Local\Temp\ipykernel_22268\702825166.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include="object")


,subscription_plan,country,favorite_genre,last_login_date
count,8160,8160,7920,7840
unique,15,26,28,3062
top,Básico,Brasil,Comedia,2026-15-03
freq,3450,1132,1112,20


En las variables de texto aparecen **muchas más categorías de las esperadas** (15 planes, 26 países, 28 géneros) y fechas con formatos raros, lo que anticipa inconsistencias de escritura.

## Valores faltantes

In [11]:
df.isnull().sum().round(2)

user_id                       0
age                           0
subscription_plan             0
monthly_watch_time_mins     193
country                       0
favorite_genre              240
last_login_date             320
customer_support_tickets      0
dtype: int64

Se confirman los faltantes: **193** en tiempo de visualización, **240** en género favorito y **320** en fecha del último ingreso. Habrá que evaluar cómo tratarlos.

## Registros duplicados

In [12]:
duplicados = df.duplicated().sum()

print(f"Cantidad de registros duplicados: {duplicados}")

Cantidad de registros duplicados: 126


Existen **126 registros completamente duplicados** (filas idénticas en todas sus columnas), candidatos a eliminar más adelante.

## Cantidad de valores únicos por variable

In [13]:
# Cantidad de valores únicos por variable
valores_unicos = pd.DataFrame({
    "Valores únicos": df.nunique()
})

valores_unicos

,Valores únicos
user_id,8000
age,69
subscription_plan,15
monthly_watch_time_mins,5788
country,26
favorite_genre,28
last_login_date,3062
customer_support_tickets,9


El conteo refuerza lo anterior: hay solo 8000 `user_id` distintos para 8160 filas (hay IDs repetidos) y demasiadas categorías en plan, país y género.

## Inconsistencias en las variables categóricas

In [14]:
columnas_categoricas = [
    "subscription_plan",
    "country",
    "favorite_genre"
]

for columna in columnas_categoricas:
    print("=" * 60)
    print(f"Variable: {columna}")
    print("=" * 60)

    print(df[columna].value_counts(dropna=False))

    print("\n")

Variable: subscription_plan
subscription_plan
Básico       3450
Estándar     2711
Premium      1519
basico         60
BASICO         52
Basic          52
básico         50
Std            48
Estándar       46
estandar       36
STANDARD       34
Premium        31
PREMIUM        26
Premiun        23
premium        22
Name: count, dtype: int64


Variable: country
country
Brasil        1132
Chile         1132
México        1129
Uruguay       1124
Perú          1120
Colombia      1116
Argentina     1087
colombia        27
méxico          25
uruguay         24
Brazil          21
COL             19
CHL             18
URY             17
MEX             16
Chile           16
argentina       16
PER             16
chile           15
Mexico          15
Peru            15
BRA             15
brasil          13
perú            12
ARG             10
Argentina       10
Name: count, dtype: int64


Variable: favorite_genre
favorite_genre
Comedia        1112
Drama          1105
Documental     1095
Thriller

Se ve con claridad el problema: una misma categoría aparece escrita de varias formas (por ejemplo *Básico, basico, BASICO, Basic*). Habrá que estandarizarlas para que cada categoría tenga una única forma.

## Análisis de `user_id` repetidos

Los duplicados completos (126) y los `user_id` repetidos no son lo mismo: puede haber usuarios con el mismo ID pero con algún dato distinto. Lo revisamos para entender de qué se trata.

### Cantidad de `user_id` repetidos

In [15]:
cantidad_ids_duplicados = df["user_id"].duplicated().sum()

print(f"Cantidad de user_id duplicados: {cantidad_ids_duplicados}")

Cantidad de user_id duplicados: 160


Hay **160 `user_id` repetidos**, más que los 126 duplicados completos. Esto confirma que algunos IDs se repiten con diferencias en alguna columna.

### Listado de registros con ID repetido

In [16]:
# IDs que aparecen más de una vez

ids_duplicados = df[df["user_id"].duplicated(keep=False)] \
                    .sort_values("user_id")

ids_duplicados

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
37,10037,33,Básico,611.0,Colombia,Documental,2019-09-29,2
8133,10037,33,Básico,611.0,Colombia,Documental,2019-09-29,2
8089,10052,25,Básico,465.7,Colombia,Acción,2022-03-31,1
52,10052,25,Básico,465.7,Colombia,Acción,2022-03-31,1
59,10059,39,Estándar,2976.6,Brasil,DRAMA,2024-06-22,1
...,...,...,...,...,...,...,...,...
8125,17784,38,Estándar,492.2,México,Documental,2021-10-04,1
8053,17902,30,Básico,505.7,Brasil,Thriller,2025-06-15,0
7902,17902,30,Básico,505.7,Brasil,THRILLER,2025-06-15,0
7994,17994,24,Básico,437.0,México,Romance,2020-08-12,0


Listamos los registros con ID repetido, ordenados, para poder compararlos entre sí.

### Comparación de casos puntuales

In [17]:
# Elegimos un user_id repetido
df[df["user_id"] == 17902]

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
7902,17902,30,Básico,505.7,Brasil,THRILLER,2025-06-15,0
8053,17902,30,Básico,505.7,Brasil,Thriller,2025-06-15,0


Usuario 17902: ambas filas son el mismo usuario; solo cambia la forma de escribir el género (*THRILLER* vs *Thriller*). Es una inconsistencia de texto, no un duplicado exacto.

In [18]:
df[df["user_id"] == 10037]

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
37,10037,33,Básico,611.0,Colombia,Documental,2019-09-29,2
8133,10037,33,Básico,611.0,Colombia,Documental,2019-09-29,2


Usuario 10037: las dos filas son **idénticas** → es un duplicado exacto.

In [19]:
df[df["user_id"] == 10059]

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
59,10059,39,Estándar,2976.6,Brasil,DRAMA,2024-06-22,1
8090,10059,39,Estándar,824.8,Brasil,Drama,2024-06-22,1


Usuario 10059: mismo ID pero con **datos distintos** (cambia el tiempo de visualización) → no debería tratarse como duplicado.

En resumen, los `user_id` repetidos no son todos iguales: hay duplicados exactos, diferencias solo de mayúsculas y casos con valores realmente distintos. Por eso esta etapa solo deja registrada la evidencia; la decisión se tomará en la limpieza.

## Conclusión de la inspección inicial

La inspección permitió conocer la estructura y el estado general del dataset **sin modificar los datos originales**. Se encontraron valores faltantes, registros duplicados, `user_id` repetidos, categorías escritas de formas distintas, fechas con formatos inconsistentes y valores numéricos fuera de rango. Estos hallazgos son la evidencia que va a orientar y justificar la etapa de limpieza y preparación del próximo notebook.

| Problema detectado        | Evidencia encontrada             | Acción prevista      |
| ------------------------- | -------------------------------- | -------------------- |
| Valores faltantes         | 193, 240 y 320 nulos             | Evaluar tratamiento  |
| Registros duplicados      | 126 registros                    | Analizar eliminación |
| User_ID repetidos         | 160 IDs repetidos                | Comparar registros   |
| Categorías inconsistentes | 15 planes, 26 países, 28 géneros | Estandarizar         |
| Fechas inválidas          | Formatos inconsistentes          | Validar              |
| Valores fuera de rango    | Edad, tiempo y tickets           | Analizar             |